# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore, process, and visualize a dataset described by a Croissant schema, using the `mlcroissant` library. All dataset entities are referenced by their unique `@id` values for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the metadata and records from the Kilifi County Mental Health Survey dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata and display summary information
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
We will first review the record sets available in this dataset. For each record set, we inspect its `@id`, name, and its contained fields (columns).

In [ ]:
# List all record sets and their details using @id references
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f" Name: {rs.name if hasattr(rs, 'name') else ''}")
    print(" Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"   - {field.id} (name: {field.name if hasattr(field, 'name') else ''}, type: {getattr(field, 'data_type', '')})")
    print('-' * 40)

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis.

> **Note:** All record sets and columns are referenced by their `@id` for clear, unambiguous access.

In [ ]:
# Extract data from each record set into a DataFrame
# Use the record set and field `@id`s listed above
dataframes = dict()
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}.")
    else:
        print(f"No records found for record set {record_set_id}.")

# Display available columns for the first record set with data
if dataframes:
    first_rs_id = next(iter(dataframes.keys()))
    print(f"\nColumns in record set {first_rs_id}:\n", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the primary survey record set (demographic and mental health indicators), referencing columns by their `@id`. We'll apply basic filtering, normalization, and grouping. *Modify `record_set_id`, `numeric_field`, and `group_field` as needed based on the column/field IDs from above!*

In [ ]:
# Set your target record set and fields by their @id (adjust as needed)
record_set_id = first_rs_id  # Use the primary record set
# Example field ids (replace with the actual field ids as per your data overview):

# Example: numeric_field='@id_of_phq9_score', group_field='@id_of_gender' (change these to your field ids)
if record_set_id in dataframes:
    df = dataframes[record_set_id]
    # Attempt to pick a numeric field and group field from columns
    possible_numeric_fields = [c for c in df.columns if 'phq' in c.lower() or 'score' in c.lower() or 'gad' in c.lower()]
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.select_dtypes(include=['number']).columns[0]
    # For grouping, try gender or similar
    possible_group_fields = [c for c in df.columns if 'gender' in c.lower() or 'sex' in c.lower()]
    group_field = possible_group_fields[0] if possible_group_fields else df.columns[0]

    print(f"Numeric field selected: {numeric_field}\nGroup field selected: {group_field}\n")

    # Filtering numeric field
    threshold = 10
    df = df[pd.to_numeric(df[numeric_field], errors='coerce').notnull()] # Filter out NaN
    filtered_df = df[df[numeric_field].astype(float) > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field, if exists
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f"mean_{numeric_field}")
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
We visualize the distribution of the selected numeric field and the group means using standard matplotlib and seaborn routines.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id in dataframes:
    sns.set_theme(style="whitegrid")

    # 1. Distribution of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].astype(float), bins=18, color='skyblue', kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # 2. Mean by group field
    if group_field in df.columns:
        plt.figure(figsize=(8, 5))
        group_means = df.groupby(group_field)[numeric_field].mean().reset_index(name=f"mean_{numeric_field}")
        sns.barplot(data=group_means, x=group_field, y=f"mean_{numeric_field}", palette='pastel')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.show()

## 6. Conclusion
- We've loaded a real-world mental health survey dataset described in the Croissant format using `mlcroissant`, referencing all entities by their unique `@id`.
- We explored the record sets and fields available, and performed basic filtering, normalization, and group-wise aggregation.
- Simple visualizations provided insight on score distributions and demographic effects.

For more advanced usage, refer to the [Croissant documentation](https://mlcommons.org/croissant/spec/) and the [mlcroissant API documentation](https://mlcommons.org/croissant/mlcroissant/).
